# Extract power doppler images and output to NIfTI
[Also see `fUSI_to_BIDS_session.py` for loading all runs in a script.]

This notebook we extract power doppler images from chestnuts output into a NifTI format using nibabel and nilearn.

It will save three files in the same folder:
1) NIFTI nii.gz containing the PD images
2) JSON for metadata ()
3) probe_events.csv for the pd h5 filename and corresponding timestamps
4) animation for quick overview

### Resources:

**Nilearn quick start** 
https://nilearn.github.io/dev/quickstart.html

**I/O**
https://nilearn.github.io/dev/manipulating_images/input_output.html#extracting-data

**Affine**
https://nipy.org/nibabel/coordinate_systems.html#the-affine-by-example

See more implementation in `SessionLoader.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import anise.utils
from anise.gui import MakeAnimation
from anise.SessionLoader import SessionLoader
from IPython.display import HTML, Video


In [3]:
#################################################################
#      Define data paths and choose output path location        #
#################################################################

save_output_on_local = False
root = Path.home() /'cassini/UCLA_collaboration/'
base_path = root / '2024-06-07/UCLA_006/'
run_id = 7
acqusition_id = 0 # switch between sequences if there are multiple (default: 0)

# define output path to save data, plots, and videos (default: /fUSI_corrected)
if save_output_on_local:
    output_path = Path.home() / "Downloads" / "UCLA_fUSI_BIDS"  # save to user defined location e.g. (/Downloads folder)
else:
    # default to save in the same root folder
    output_path = root / "UCLA_fUSI_BIDS"

# Find path using sessionLoader
ses = SessionLoader(root, base_path, run_id, acqusition_id, output_path=output_path)

Selected sequence in the acquisitions path:
--> 0. seq-RF_full_apert_5000000Hz_5a_3c_2rp_10Dmin_40Dmax_3el_TGC0ini_50end_8000Tdur_-1Gain_3AFE_200loops

Loaded directories:

 - log_file_path:       	/home/ewina/cassini/UCLA_collaboration/2024-06-07/UCLA_006/run-07/logs/task_1.log
 - task_event_file_path:  	/home/ewina/cassini/UCLA_collaboration/2024-06-07/UCLA_006/run-07/streams/task_1-event_stream_085932.h5
 - probe_event_file_path: 	/home/ewina/cassini/UCLA_collaboration/2024-06-07/UCLA_006/run-07/streams/probe_1-event_stream_085932.h5
 - sequence_data_path:  	/home/ewina/cassini/UCLA_collaboration/2024-06-07/UCLA_006/run-07/acquisitions/seq-RF_full_apert_5000000Hz_5a_3c_2rp_10Dmin_40Dmax_3el_TGC0ini_50end_8000Tdur_-1Gain_3AFE_200loops
 - raw_data_path:       	/home/ewina/cassini/UCLA_collaboration/2024-06-07/UCLA_006/run-07/acquisitions/seq-RF_full_apert_5000000Hz_5a_3c_2rp_10Dmin_40Dmax_3el_TGC0ini_50end_8000Tdur_-1Gain_3AFE_200loops/raw_frame_data
 - beamformed_path:     	/home/ewi

In [28]:
#################################################################
#                Load power doppler from h5                     #
#################################################################

# process power doppler files based on same number of tissue components in clutter filter (PCA)
unique_num_tissue_components = ses.find_unique_num_tissue_components_within_single_acqusition()

# Parse separately if more than one tissue components within a single acqusition
for n_tc in unique_num_tissue_components:
    try:
        ses.filter_power_doppler_files(num_tissue_components=n_tc)
        
        # load fusi data
        fusi_data = ses.load_fusi_frames()

    except:
        continue

Scanning through power doppler folder...
Unique number of tissue components found: {'50', '60'}

Number of file with the specified number of tissue components: 94/504

Matching and adding filenames to probe_events...
No entries in 'fusi_file_name' is populated with number of tissue components=50

Number of file with the specified number of tissue components: 410/504

Matching and adding filenames to probe_events...
All entries in 'fusi_file_name' are properly populated.
Loading dataset from disk to memory
Loaded. releasing dataset_on_disk open-file

Loaded in fusi_data with shape: (274, 2, 225, 410)


In [26]:
# ses.filter_power_doppler_files(num_tissue_components='50')
# # load fusi data
# # fusi_data = ses.load_fusi_frames()
# ses.probe_events
# print(ses.power_doppler_df["Filename"].values)
ses.probe_events['fusi_file_name'].isnull().all()

True

In [ ]:
#################################################################
#         Load task event and add timing offset                 #
#################################################################

# task_name = '' # name can be specified in extract_task_events(), otherwise automatically detected (e.g. audio, light, SSEP)
# task_description = '' # description can be specified in extract_task_events(), otherwise automatically added
ses.extract_task_events()

# calculate offset between task start time and ensemble start time
behavior_offset = (ses.probe_events['global_start_time'][0] - ses.task_start).total_seconds()
print('\n\tpwd ensemble start time:', ses.probe_events['global_start_time'][0])
print('-\ttask start time: \t', ses.task_start)
print('___________________________________________________________')
print(f'Adjusting behavioral offset by \t\t   {behavior_offset} seconds')

# adjust task events onset time
ses.task_events['onset'] = ses.task_events['onset'] - behavior_offset
ses.task_events


In [ ]:
#################################################################
#                          save outputs                         #
#################################################################

fus_dir = ses.output_path / 'sourcedata' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}' / 'fus'
beh_dir = ses.output_path / 'sourcedata' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}' / 'beh'

# save to NIFTI and metadata output_path
_, filename = ses.save_to_nifti(fusi_data, output_path=fus_dir)

# save task events (need to run after save_to_nifti for self.output_filename)
ses.save_task_events(output_path=beh_dir)


In [ ]:
#################################################################
#           Display and save power doppler movie                #
#################################################################

ani = None
file_path = ses.output_path / 'derivatives' / 'registration' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}'
file_path.mkdir(parents=True, exist_ok=True)
if fusi_data is not None:
    ani = MakeAnimation(fusi_data[:, 0], 
                        output_file=str( file_path / f'{filename}_before.mp4'),  # mp4
                        fps=10, 
                        depth=ses.sidecar['Depth'], 
                        lateral=ses.sidecar['Lateral'], 
                        time=ses.sidecar['VolumeTiming'],
    )

    ani = MakeAnimation(fusi_data[:, 0],
                        output_file=str(file_path / f'{filename}_before.gif'), # gif
                        fps=10, 
                        depth=ses.sidecar['Depth'], 
                        lateral=ses.sidecar['Lateral'], 
                        time=ses.sidecar['VolumeTiming'],
    )
HTML(ani.to_jshtml())